<a href="https://colab.research.google.com/github/anamacao/FAPESP-PIBIC-scrapping/blob/main/UE/database_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Manipulação & Análise do Banco SQLite

Use o arquivo `internet_governance_news.db` gerado pelos coletores na mesma sessão do Colab. A rotina de limpeza está definida na última célula, mas sua chamada permanece comentada.


In [ ]:
# %%
import sqlite3
import pandas as pd
import re
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "notebook_connected"

print("✅ Imports prontos")


In [ ]:
# %%
DATABASE_NAME = "internet_governance_news.db"
print(f"📦 Banco configurado: {DATABASE_NAME}")


### Ver tabelas existentes

In [ ]:
# %%
def list_tables():
    conn = sqlite3.connect(DATABASE_NAME)
    df = pd.read_sql("""
        SELECT name FROM sqlite_master WHERE type='table'
    """, conn)
    conn.close()
    return df

list_tables()


### Visualizar estrutura da tabela articles

In [ ]:
# %%
def table_schema():
    conn = sqlite3.connect(DATABASE_NAME)
    df = pd.read_sql("PRAGMA table_info(articles)", conn)
    conn.close()
    return df

table_schema()


### Carregar dados do banco

In [ ]:
# %%
def load_articles():
    conn = sqlite3.connect(DATABASE_NAME)
    try:
        existe = conn.execute(
            "SELECT 1 FROM sqlite_master WHERE type = 'table' AND name = 'articles'"
        ).fetchone()
        if existe:
            local = pd.read_sql('SELECT * FROM articles ORDER BY date DESC', conn)
            if not local.empty:
                print("📦 Fonte: SQLite desta sessão")
                return local
        # O Colab reinicia o disco de trabalho entre sessões. O CSV versionado
        # mantém a análise independente da ordem em que os coletores rodaram.
        url = "https://raw.githubusercontent.com/anamacao/FAPESP-PIBIC-scrapping/main/data/news.csv"
        news = pd.read_csv(url, dtype=str, keep_default_na=False)
        required = {"title", "published_date", "url", "source_id"}
        if not required.issubset(news.columns):
            raise ValueError(f"Colunas ausentes no CSV: {sorted(required - set(news.columns))}")
        news = news.drop_duplicates(subset=["source_id", "url"]).copy()
        news = news.rename(columns={"published_date": "date"})
        source_names = {"nic": "NIC.br", "edpb": "EDPB", "mercociudades": "Mercociudades", "senado_federal": "Agência Senado"}
        news["source"] = news["source_id"].map(source_names).fillna(news["source_id"])
        news["author"] = news["source"]
        news["id"] = range(1, len(news) + 1)
        print("📦 Fonte: CSV acumulado do GitHub")
        return news
    finally:
        conn.close()

df = load_articles()
print(f"📊 Total de registros: {len(df)}")
df.head()


### Filtrar por fonte (Senado, Câmara, CGI…)

In [ ]:
# %%
def filter_by_source(df, source):
    return df[df["source"] == source].copy()

df_senado = filter_by_source(df, "Agência Senado")
print(f"🏛️ Senado: {len(df_senado)} registros")


### Apagar TODOS os dados da tabela (reset total)

In [ ]:
# %%
def clear_articles():
    conn = sqlite3.connect(DATABASE_NAME)
    cursor = conn.cursor()
    cursor.execute("DELETE FROM articles")
    conn.commit()
    conn.close()
    print("🧹 Tabela articles limpa!")

# clear_articles()  # ← descomente se quiser apagar tudo
